# Theory and Implementation of Time-Series Similarity and Clustering on Gravitational Wave Data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kojikoji81/DTW_GravitationalWaves/blob/main/DTW_GravitationalWaves_Notebook.ipynb)

This notebook provides a detailed mathematical and physical background, along with Python implementations, for extracting, preprocessing, and clustering gravitational wave (GW) time-series data from laser interferometer detectors (LIGO/Virgo/KAGRA).

Specifically, we explore the following key topics:
1. **Gravitational Wave Signal Processing (Bandpass Filtering & Whitening)**
2. **Time-Series Distance Metrics (Euclidean Distance, Dynamic Time Warping (DTW), Shape-Based Distance (SBD))**
3. **Time-Series Clustering Algorithms (TimeSeriesKMeans, K-Shape)**

Finally, we demonstrate these techniques using a simulated GW dataset containing chirp signals, sinusoidal noise, and white noise to compare the performance of each clustering approach.



In [ ]:
# If you are running this notebook on Google Colab, execute this cell to install the required libraries.
# If running in a local environment, you can skip this cell.
import sys
if 'google.colab' in sys.modules:
    print("Google Colab detected. Installing required libraries...")
    !pip install gwpy tslearn plotly scikit-learn tqdm scipy
else:
    print("Local environment detected. Skipping automatic library installation.")



## 1. Gravitational Wave Signal Processing: Physics & Mathematics

The raw strain time-series data $s(t)$ recorded by a gravitational wave detector is a linear combination of a tiny gravitational wave signal $h(t)$ and a massive detector noise $n(t)$:

$$
s(t) = h(t) + n(t)
$$

Because the amplitude of $h(t)$ is extremely small (on the order of $10^{-21}$), it is completely buried under various sources of instrumental and environmental noise $n(t)$. Thus, rigorous preprocessing is indispensable before applying any pattern recognition.

### 1.1 Bandpass Filtering
To block out dominant low-frequency noise (e.g., ground seismic motion) and high-frequency noise (e.g., photon shot noise in the lasers), we apply a bandpass filter. It passes frequencies between the lower and upper cutoffs $f\_L, f\_H$, matching the detector's most sensitive band (typically tens to hundreds of Hz).

### 1.2 Mathematical Formulation of Whitening
Gravitational wave noise $n(t)$ is "colored noise," meaning its noise power varies drastically across different frequencies. Whitening is the process of flattening this frequency-dependent variance, transforming the colored noise into statistical "white noise" (which has uniform power at all frequencies).

In a stationary stochastic process, the noise characteristics are defined by the **one-sided Power Spectral Density (PSD)** $S\_n(f)$.
Given the Fourier transform of the strain $\tilde{s}(f)$, the whitened strain $\tilde{s}\_{\text{white}}(f)$ in the frequency domain is defined by dividing the Fourier transform by the **Amplitude Spectral Density (ASD)** $\sqrt{S\_n(f)}$:

$$
\tilde{s}\_{\text{white}}(f) = \frac{\tilde{s}(f)}{\sqrt{S\_n(f)}}
$$

Applying the inverse Fourier transform $\mathcal{F}^{-1}$ yields the whitened time-series strain $s\_{\text{white}}(t)$:

$$
s\_{\text{white}}(t) = \mathcal{F}^{-1} \left( \frac{\tilde{s}(f)}{\sqrt{S\_n(f)}} \right)
$$

#### [Detailed Mathematical Proof of Noise Flattening]
Let us verify mathematically that the whitened data indeed exhibits white noise properties (i.e., a flat PSD equal to 1).
By definition of the one-sided PSD, the ensemble average of the raw noise Fourier components $\tilde{s}(f)$ is given by:

$$
\langle \tilde{s}(f) \tilde{s}^*(f') \rangle = \frac{1}{2} S\_n(f) \delta(f - f')
$$

where $\delta$ is the Dirac delta function and $*$ denotes the complex conjugate.
Now, we compute the covariance of the whitened components:

$$
\langle \tilde{s}\_{\text{white}}(f) \tilde{s}\_{\text{white}}^*(f') \rangle = \left\langle \frac{\tilde{s}(f)}{\sqrt{S\_n(f)}} \frac{\tilde{s}^*(f')}{\sqrt{S\_n(f')}} \right\rangle
$$

$$
\langle \tilde{s}\_{\text{white}}(f) \tilde{s}\_{\text{white}}^*(f') \rangle = \frac{1}{\sqrt{S\_n(f) S\_n(f')}} \langle \tilde{s}(f) \tilde{s}^*(f') \rangle
$$

$$
\langle \tilde{s}\_{\text{white}}(f) \tilde{s}\_{\text{white}}^*(f') \rangle = \frac{1}{\sqrt{S\_n(f) S\_n(f')}} \left( \frac{1}{2} S\_n(f) \delta(f - f') \right)
$$

Using the identity of the Dirac delta function (which is non-zero only when $f = f'$), we substitute $S\_n(f')$ with $S\_n(f)$ in the denominator:

$$
\langle \tilde{s}\_{\text{white}}(f) \tilde{s}\_{\text{white}}^*(f') \rangle = \frac{1}{2} \frac{S\_n(f)}{S\_n(f)} \delta(f - f')
$$

$$
\langle \tilde{s}\_{\text{white}}(f) \tilde{s}\_{\text{white}}^*(f') \rangle = \frac{1}{2} \delta(f - f')
$$

Optionally normalising the constant shows that the PSD of the whitened strain becomes constant over all frequencies, confirming that the colored noise has been successfully flattened into white noise.



## 2. Time-Series Distance Metrics

We evaluate and compare three distance metrics for measuring the similarity between two time-series sequences $X = (x\_1, x\_2, \dots, x\_N)$ and $Y = (y\_1, y\_2, \dots, y\_M)$.

### 2.1 Euclidean Distance
Treating time-series as vectors of equal length ($N=M$), the Euclidean distance is the square root of the sum of squared differences:

$$
d\_{\text{Euclidean}}(X, Y) = \sqrt{\sum\_{i=1}^{N} (x\_i - y\_i)^2}
$$

**Limitation**: Euclidean distance is extremely sensitive to minor shifts along the time axis (phase shifts). Even if two waveforms share an identical shape, a slight offset in time will cause Euclidean distance to classify them as entirely different.

### 2.2 Dynamic Time Warping (DTW)
DTW allows for non-linear warping along the time axis to find an optimal alignment (warping path) between two sequences.

#### 2.2.1 DP Recursion Formulation
Let the local cost between points be $d(x\_i, y\_j) = (x\_i - y\_j)^2$.
The cumulative distance matrix $\gamma \in \mathbb{R}^{N \times M}$ is computed recursively using Dynamic Programming (DP):

$$
\gamma(i, j) = d(x\_i, y\_j) + \min \left\{ \gamma(i-1, j), \gamma(i, j-1), \gamma(i-1, j-1) \right\}
$$

subject to the boundary conditions:

$$
\gamma(1, 1) = d(x\_1, y\_1)
$$

$$
\gamma(i, 1) = \gamma(i-1, 1) + d(x\_i, y\_1) \quad (\text{for } i > 1)
$$

$$
\gamma(1, j) = \gamma(1, j-1) + d(x\_1, y\_j) \quad (\text{for } j > 1)
$$

The final DTW distance is defined as:

$$
d\_{\text{DTW}}(X, Y) = \gamma(N, M)
$$

### 2.3 Shape-Based Distance (SBD)
SBD measures similarity based on cross-correlation, allowing for arbitrary time shifts (lags).

Given two standardized sequences $x, y$ of length $m$ (with L2 norms normalized to 1: $\|x\|\_2 = \|y\|\_2 = 1$), the cross-covariance $R\_q(x, y)$ at lag $q \in [-m, m]$ is defined as:

$$
R\_q(x, y) = \begin{cases} \sum\_{i=1}^{m-q} x\_{i+q} y\_i & (q \ge 0) \\ R\_{-q}(y, x) & (q < 0) \end{cases}
$$

The Normalized Cross-Correlation (NCC) is:

$$
NCC\_q(x, y) = \frac{R\_q(x, y)}{\|x\|\_2 \|y\|\_2} = R\_q(x, y)
$$

SBD is then defined by finding the optimal lag $q$ that maximizes NCC:

$$
SBD(x, y) = 1 - \max\_{q} NCC\_q(x, y)
$$

SBD ranges in $[0, 2]$, where $0$ indicates that the sequences are identical up to a time shift.

#### [FFT Acceleration via Convolution Theorem]
While a naive computation of $R\_q(x, y)$ takes $O(m^2)$ time, we can accelerate it to $O(m \log m)$ using the Fast Fourier Transform (FFT).
By the convolution theorem, the cross-correlation in the time domain corresponds to the product of one Fourier transform and the complex conjugate of the other in the frequency domain:

$$
R(x, y) = \mathcal{F}^{-1} \left( \mathcal{F}(x) \cdot \mathcal{F}(y)^* \right)
$$

where $\mathcal{F}$ is the Discrete Fourier Transform, $\mathcal{F}^{-1}$ is the inverse DFT, and $*$ is the complex conjugate. This enables extremely fast SBD computations even for large datasets.



## 3. Time-Series Clustering Algorithms

### 3.1 TimeSeriesKMeans
This is an extension of the standard K-Means algorithm where the distance metric is replaced with DTW (or Soft-DTW).
- **DTW**: Robust but computationally expensive due to the iterative Barycenter Averaging (DBA) needed to update centroids.
- **Soft-DTW**: Uses a soft-minimum version of DTW, making the distance function differentiable. This allows direct centroid optimization via gradient descent, yielding smooth average waveforms.

### 3.2 K-Shape Algorithm
K-Shape is a highly efficient clustering method that uses SBD as its distance metric and updates cluster centroids by formulating an optimization problem solved via eigenvalue decomposition.

#### [Derivation of K-Shape Centroid Update via Eigenvalue Problem]
Let $\mu\_k$ be the centroid of cluster $C\_k$. The objective is to find $\mu\_k$ that maximizes the sum of squared cross-correlations with all aligned sequences $x\_i \in C\_k$:

$$
\arg\max\_{\mu\_k} \sum\_{x\_i \in C\_k} \left( \max\_{q} NCC\_q(x\_i, \mu\_k) \right)^2
$$

Let $x'\_i$ be the sequence $x\_i$ optimally aligned (phase-shifted) to the current centroid $\mu\_k$.
Under this alignment, the maximization problem simplifies to:

$$
\arg\max\_{\mu\_k} \sum\_{x\_i \in C\_k} \left( \frac{\mu\_k^T x'\_i}{\|\mu\_k\|\_2 \|x'\_i\|\_2} \right)^2
$$

Since all sequences are standardized and normalized, we have $\|x'\_i\|\_2 = 1$. Thus:

$$
\arg\max\_{\mu\_k} \sum\_{x\_i \in C\_k} \frac{(\mu\_k^T x'\_i)^2}{\mu\_k^T \mu\_k}
$$

Expanding the numerator:

$$
(\mu\_k^T x'\_i)^2 = (\mu\_k^T x'\_i) (x'\_i^T \mu\_k) = \mu\_k^T (x'\_i x'\_i^T) \mu\_k
$$

Substituting this back into the summation yields:

$$
\sum\_{x\_i \in C\_k} (\mu\_k^T x'\_i)^2 = \sum\_{x\_i \in C\_k} \mu\_k^T (x'\_i x'\_i^T) \mu\_k = \mu\_k^T \left( \sum\_{x\_i \in C\_k} x'\_i x'\_i^T \right) \mu\_k
$$

We define the characteristic covariance-like matrix $M$ for the cluster:

$$
M = \sum\_{x\_i \in C\_k} x'\_i x'\_i^T
$$

The optimization problem can now be written as:

$$
\arg\max\_{\mu\_k} \frac{\mu\_k^T M \mu\_k}{\mu\_k^T \mu\_k}
$$

This expression is the **Rayleigh quotient** of the matrix $M$.
According to the properties of the Rayleigh quotient, the vector $\mu\_k$ that maximizes this ratio is the **principal eigenvector** corresponding to the **largest eigenvalue** of the matrix $M$.

By reducing the centroid update step to a standard eigenvalue problem, K-Shape avoids computationally expensive iterative optimizations, resulting in outstanding execution speeds.



## 4. Implementation Demo

Let us now execute time-series clustering on simulated gravitational wave data.

### 4.1 Importing Libraries



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from gwpy.timeseries import TimeSeries
from tslearn.clustering import KShape, TimeSeriesKMeans
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')



### 4.2 Simulating Gravitational Wave Signal + Noise
We generate a synthetic strain dataset containing a chirp signal (inspiraling binary merger), low-frequency sine waves, high-frequency sine waves, and white noise.



In [ ]:
# Configuration parameters
sample_rate = 2048  # Hz
duration = 8.0      # seconds
t0 = 1126259462
times = np.linspace(0, duration, int(sample_rate * duration))

# Base white noise
signal = np.random.randn(len(times)) * 0.5

# 1. Inject low-frequency sine wave (35 Hz) at 1.5 - 2.5 seconds
idx1 = (times >= 1.5) & (times < 2.5)
signal[idx1] += np.sin(2 * np.pi * 35 * times[idx1]) * 1.5

# 2. Inject chirp signal (frequency sweeping from 30Hz to 150Hz) at 3.5 - 4.5 seconds
idx2 = (times >= 3.5) & (times < 4.5)
t_chirp = times[idx2] - 3.5
f_chirp = 30 + (150 - 30) * (t_chirp / 1.0)
signal[idx2] += np.sin(2 * np.pi * f_chirp * t_chirp) * 2.5

# 3. Inject high-frequency sine wave (120 Hz) at 5.5 - 6.5 seconds
idx3 = (times >= 5.5) & (times < 6.5)
signal[idx3] += np.sin(2 * np.pi * 120 * times[idx3]) * 1.2

# Create TimeSeries object
data = TimeSeries(signal, sample_rate=sample_rate, t0=t0, unit='strain')

# Visualize raw data
plt.figure(figsize=(12, 4))
plt.plot(data.times.value - t0, data.value, color='black', alpha=0.8)
plt.title("Simulated Gravitational Wave Data (Signal + Noise)")
plt.xlabel("Time (seconds)")
plt.ylabel("Strain")
plt.grid(True, linestyle=':')
plt.show()



### 4.3 Preprocessing and Sequence Segmentation
We apply a bandpass filter (30-300Hz), split the data into 0.5-second non-overlapping windows, and standardize each segment.



In [ ]:
# Apply bandpass filter
data_filtered = data.bandpass(30, 300)

# Windowing parameters
win_len_sec = 0.5
step_len_sec = 0.5
win_samples = int(win_len_sec * sample_rate)
step_samples = int(step_len_sec * sample_rate)

sequences = []
times_labels = []

for i in range(0, len(data_filtered) - win_samples + 1, step_samples):
    seq = data_filtered[i : i + win_samples].value
    t_start = data_filtered.t0.value + (i / sample_rate)
    
    # Standardize to bring mean to 0 and variance to 1
    if np.std(seq) > 1e-6:
        seq_norm = StandardScaler().fit_transform(seq.reshape(-1, 1)).flatten()
        sequences.append(seq_norm)
        times_labels.append(t_start)

X = np.array(sequences)
times_labels = np.array(times_labels)
print(f"Generated {X.shape[0]} sequences, each of length {X.shape[1]} samples.")



### 4.4 Running Clustering Algorithms
We run KShape (SBD), TimeSeriesKMeans (DTW), and standard TimeSeriesKMeans (Euclidean) on the extracted sequences.



In [ ]:
n_clusters = 3

# 1. K-Shape
print("1. Running K-Shape clustering...")
kshape = KShape(n_clusters=n_clusters, random_state=42, verbose=False, max_iter=10)
kshape.fit(X)
labels_kshape = kshape.labels_
centroids_kshape = kshape.cluster_centers_

# 2. TimeSeriesKMeans (DTW)
print("2. Running TimeSeriesKMeans (DTW)...")
kmeans_dtw = TimeSeriesKMeans(n_clusters=n_clusters, metric="dtw", random_state=42, verbose=False, max_iter=5, n_jobs=-1)
kmeans_dtw.fit(X)
labels_dtw = kmeans_dtw.labels_
centroids_dtw = kmeans_dtw.cluster_centers_

# 3. TimeSeriesKMeans (Euclidean)
print("3. Running TimeSeriesKMeans (Euclidean)...")
kmeans_eucl = TimeSeriesKMeans(n_clusters=n_clusters, metric="euclidean", random_state=42, verbose=False, max_iter=10)
kmeans_eucl.fit(X)
labels_eucl = kmeans_eucl.labels_
centroids_eucl = kmeans_eucl.cluster_centers_



### 4.5 Visualizing and Comparing Results
We plot the clustered sequences and their respective centroids for each of the three algorithms.



In [ ]:
def plot_clustering_results(X_seq, labels, centroids, method_name):
    fig, axes = plt.subplots(n_clusters, 1, figsize=(10, 2.5 * n_clusters), sharex=True)
    colors = plt.cm.get_cmap('Set1', n_clusters)
    
    for yi in range(n_clusters):
        ax = axes[yi]
        cluster_color = colors(yi)
        indices = np.where(labels == yi)[0]
        
        # Plot individual sequences in the cluster
        for idx in indices:
            ax.plot(X_seq[idx].ravel(), color=cluster_color, alpha=0.3, linewidth=1)
            
        # Plot centroid
        ax.plot(centroids[yi].ravel(), color=cluster_color, linewidth=2.5, linestyle='--',
                label=f"Centroid (Count: {len(indices)})")
        
        ax.set_ylabel("Std Strain")
        ax.set_title(f"Cluster {yi + 1}")
        ax.legend(loc='upper right')
        ax.grid(True, linestyle=':', alpha=0.5)
        
    plt.xlabel("Sample Index")
    plt.suptitle(f"Clustering Results using {method_name}", y=1.02, fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Display plots
plot_clustering_results(X, labels_eucl, centroids_eucl, "Euclidean Distance")
plot_clustering_results(X, labels_dtw, centroids_dtw, "Dynamic Time Warping (DTW)")
plot_clustering_results(X, labels_kshape, centroids_kshape, "K-Shape (SBD)")



### 4.6 Discussion & Analysis
From the plots, we can draw the following conclusions:
- **Euclidean Distance**: Struggles with phase-shifted signals. It often splits identical wave structures into separate clusters or mixes noise with genuine signals.
- **Dynamic Time Warping (DTW)**: Captures features robustly by aligning temporal expansions, which is useful for frequency-sweeping signals. However, it has high computational overhead.
- **K-Shape (SBD)**: Successfully separates the synthetic data into clean, physically distinct clusters: "Chirp signal," "Low-frequency sine," and "Pure noise." By utilizing cross-correlation and FFT, it achieves both superior shape recognition and execution speed.

